# Sales Analytics Pipeline | Python & Pandas

**Objective:** Transform a raw Pakistan e-commerce transactions CSV into analysis-ready sales, product, customer, and date tables.

**Workflow:** Load → inspect → standardize → validate → filter → calculate revenue → build tables → export CSVs.

> This notebook demonstrates a local/Colab Pandas ETL workflow. It does **not** connect to PostgreSQL or implement automated scheduling. Results depend on the source CSV and the chosen order-status rules.

### Input
Place `Pakistan Largest Ecommerce Dataset1 (1).csv` in the notebook's working directory, or update `DATA_PATH` below. The original Colab version used `/content/Pakistan Largest Ecommerce Dataset1 (1).csv`.

### Output
`sales.csv`, `products.csv`, `customers.csv`, `date_dim.csv`.

**Business rules to review:** Revenue is defined here as `qty_ordered × price` (before discount, tax, and shipping); statuses `complete`, `received`, `paid`, `closed`, and `processing` are included, so results should not be described as exclusively finalized sales. Duplicate removal uses `order_id`, `sku`, and `qty_ordered` and may remove legitimate repeated lines; inspect this rule before production use.

## 1. Load source data

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('Pakistan Largest Ecommerce Dataset1 (1).csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing source CSV: {DATA_PATH.resolve()}. Upload it or update DATA_PATH.')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded: {len(df):,} rows × {len(df.columns)} columns')

## 2. Explore source data
Review data types, descriptive statistics, and exact duplicate rows before cleaning.

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.describe(include=["object", "bool"])

In [ ]:
df.loc[df.duplicated()]

## 3. Standardize fields and data types

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)
df.rename(columns={'increment_id':'order_id','category_name_1':'category','discount_amount':'discount'}, inplace=True)
drop_cols = ['unnamed:_21','unnamed:_22','unnamed:_23','unnamed:_24','unnamed:_25',
             'working_date','bi_status','mv','year','month','m-y','fy','sales_commission_code']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
required = ['order_id','item_id','created_at','customer_id','sku','qty_ordered','price',
            'discount','payment_method','status','category','customer_since']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Required columns missing from input: {missing}')
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['customer_since'] = pd.to_datetime(df['customer_since'], errors='coerce')
for col in ['price','qty_ordered','discount']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['customer_id'] = pd.to_numeric(df['customer_id'], errors='coerce').astype('Int64')
print('Standardized columns and data types.')

## 4. Clean and filter records
The status filter and duplicate key are project assumptions, not universally correct accounting rules.

In [ ]:
before = len(df)
df.dropna(subset=['order_id','sku','created_at','price','qty_ordered'], inplace=True)
df['discount'] = df['discount'].fillna(0)
df['status'] = df['status'].astype('string').str.strip().str.lower()
valid_statuses = ['complete','received','paid','closed','processing']
df = df[df['status'].isin(valid_statuses)].copy()
possible_duplicates = df.duplicated(subset=['order_id','sku','qty_ordered']).sum()
print(f'Potential repeated line items by selected key: {possible_duplicates:,}')
df.drop_duplicates(subset=['order_id','sku','qty_ordered'], keep='first', inplace=True)
df['revenue'] = df['qty_ordered'] * df['price']
print(f'Rows before cleaning: {before:,}; rows after: {len(df):,}')

## 5. Create analysis-ready tables

In [ ]:
if df.empty:
    raise ValueError('No rows remain after filtering; inspect statuses and source data.')
sales = df[['order_id','item_id','created_at','customer_id','sku','qty_ordered',
            'price','revenue','discount','payment_method','status']].copy()
products = df.groupby('sku', as_index=False).agg(category=('category','first'), price=('price','median'))
customers = (df.dropna(subset=['customer_id']).groupby('customer_id', as_index=False)
             .agg(customer_since=('customer_since','first'), first_order_date=('created_at','min'),
                  last_order_date=('created_at','max'), total_orders=('order_id','nunique'),
                  total_spent=('revenue','sum')))
all_dates = pd.date_range(df['created_at'].min().normalize(), df['created_at'].max().normalize(), freq='D')
date_dim = pd.DataFrame({'date':all_dates,'year':all_dates.year,'month':all_dates.month,
                         'month_name':all_dates.strftime('%B'),'quarter':all_dates.quarter})
for name, table in [('sales',sales),('products',products),('customers',customers),('date_dim',date_dim)]:
    print(f'{name}: {len(table):,} rows')
    display(table.head(3))

## 6. Validate and export
Check basic output properties before writing the four CSV tables.

In [ ]:
assert sales['revenue'].notna().all(), 'Null revenue values found'
assert products['sku'].is_unique, 'Product keys are not unique'
assert customers['customer_id'].is_unique, 'Customer keys are not unique'
assert date_dim['date'].is_unique, 'Date keys are not unique'
for name, table in [('sales',sales),('products',products),('customers',customers),('date_dim',date_dim)]:
    table.to_csv(f'{name}.csv', index=False)
    print(f'Saved {name}.csv ({len(table):,} rows)')

## Limitations and next steps
- Inspect the selected status filter and duplicate key against business definitions before operational use.
- The `products` table chooses the first category and median price per SKU; this is a simplified dimension.
- Revenue is gross line-item value and does not subtract discounts or incorporate refunds.
- Future extensions: SQL Server/PostgreSQL loading, unit tests, configuration, logging, and scheduled execution.